# **CatBoost**

In [1]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [2]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
from catboost import CatBoostClassifier

def train_catboost_ttp(df, train_size, test_size, step):

    splitter = prep(
        df=df,
        target_fn=ttp_target,
        target_name="ttp",
        target_col="TTP_class",
        horizons=[12, 24, 48],
        train_size=train_size,
        test_size=test_size,
        step=step,
        target_kwargs={"n_classes": 3},
        scale_cols=[
            "Open", "High", "Low", "Close",
            "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
            "AO",
            "AddOn_Anchor_Level", "AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:

        model = CatBoostClassifier(
            loss_function="MultiClass",
            classes_count=3,
            iterations=300,
            depth=5,
            learning_rate=0.05,

            bootstrap_type="Bernoulli",
            subsample=0.8,
            rsm=0.8,

            random_seed=42,
            verbose=False
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test).reshape(-1)

        metrics = merged_metrics(y_test, y_pred)

        append_results({
            "task_type": "classification",
            "model_name": "CatBoost",
            "model_family": "catboost",
            "model_params": {
                "iterations": 300,
                "depth": 5,
                "learning_rate": 0.05,
                "bootstrap_type": "Bernoulli",
                "subsample": 0.8,
                "rsm": 0.8
            },
            "target_name": "ttp",
            "target_variant": "3class",
            "horizons": "12_24_48",
            **metrics
        })

In [4]:
train_catboost_ttp(
    df=df,
    train_size=1000,
    test_size=200,
    step=100
)